# 5장 2강. 분석 결과 평가와 개선
## 실습문제


## 실습 목표

- 정답·원문과 비교하는 정확성과 반복 결과를 비교하는 일관성을 구분할 수 있다.
- 분류 정확도(Accuracy)를 계산하고, 오답이 발생한 상담을 확인할 수 있다.
- 정보 추출의 필드별 정확성과 요약의 사실 일치·핵심 정보 유지·불필요한 정보 생성을 평가할 수 있다.
- 동일한 평가 데이터로 프롬프트와 워크플로우의 가상 결과를 비교할 수 있다.
- 개선된 오류·남은 오류·새로운 오류를 구분하고 근거에 맞는 개선 방향을 제안할 수 있다.

## 실습 환경 / 데이터

- Python 3, Jupyter Notebook 또는 VS Code 노트북, pandas
- CSV 3개를 노트북과 같은 폴더에 두고, 다른 위치라면 `DATA_DIR`를 수정합니다.
- 이 파일의 준비 코드부터 차례대로 실행합니다. 이전 강의 노트북은 필요하지 않습니다.

| 파일 | 이번 실습에서의 역할 |
|---|---|
| 엘지유플러스_분류.csv | 상담 요건의 평가 정답·반복 결과·프롬프트 비교 |
| 엘지유플러스_요약.csv | 기존 요약 참고 및 가상 요약의 사실·핵심 정보 검토 |
| 엘지유플러스_질의응답.csv | 질문별 기대 답변을 원문에 대조하여 정보 추출 평가 |

원본 컬럼은 source_id, consulting_category, task_category, instruction, input, output입니다. source_id는 상담 ID, instruction은 지시·질문, input은 원문, output은 CSV에 이미 저장된 출력입니다. source_id 하나에 여러 작업이 있으므로 세부 작업과 질문도 지정합니다.

### 실습 범위와 자료 구분

교안의 명언·리뷰 사례를 상담 텍스트에 적용합니다. **필수 1·필수 2·과제 1**, 기초~중급 수준이며 과제도 필수와 같은 수준입니다.

- 원문·지시·기존 출력은 실제 CSV 데이터입니다. 기존 output을 무조건 정답으로 간주하지 않고 원문과 명시한 기준으로 확인합니다.
- 모든 새 분류·추출·요약 결과와 반복 실행 기록은 **작성자가 만든 가상 평가 자료**입니다. 실제 API를 실행하거나 모델 성능을 측정한 자료가 아닙니다.
- 같은 평가 데이터로 비교하는 계산과 해석을 연습합니다. 코드가 생성 결과의 의미를 자동으로 이해하고 평가하지는 않습니다.
- 원문 근거가 필요한 판단은 사람이 읽고 표에 작성합니다. 그 수동 판단을 파이썬으로 정리·집계합니다.
- API 호출·LLM 평가 모델·복잡한 평가 시스템·교안에 없는 평가 지표는 사용하지 않습니다.
- 가림 표시를 추측하여 복원하지 않습니다. 상담 내용은 해당 원문의 사실로 평가합니다.

| 교안 내용 | 실습 |
|---|---|
| 정확성과 일관성 | 필수 1: 분류 정확도와 3회 결과 비교 |
| 작업별 평가 기준 | 필수 2: 정보 추출의 필드와 요약의 내용 검토 |
| 프롬프트·워크플로우 비교 | 과제 1: 같은 문서·정답·기준으로 가상 결과 비교 |
| 오류 분석과 개선 검증 | 과제 1: 개선·악화·남은 오류와 추가 확인 계획 |


## 실습 준비

제공 코드를 실행하세요. 함수 구현 자체는 평가하지 않습니다.

1. **데이터 불러오기:** 세 파일을 별도 표로 읽습니다.
2. **분석 대상 컬럼 확인:** 원문·지시·기존 출력과 상담 ID를 확인합니다.
3. **결측치 및 자료형 확인:** 원본 컬럼의 자료형, 결측치와 빈 문자열을 확인합니다.
4. **기초 통계량과 분포 확인:** 고유 상담 수, 작업 유형별 행 수와 본문 길이를 확인합니다.

record_id는 파일 이름과 현재 행 위치로 만든 실습용 ID이며 원본의 영구 식별자는 아닙니다. 본문 글자 수는 토큰 수가 아닙니다. 파일의 작업 행 수가 다르다는 이유만으로 LLM의 결과 누락이라고 판단하지 않습니다.


In [9]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(".")
classification = pd.read_csv(DATA_DIR / "엘지유플러스_분류.csv")
summarization = pd.read_csv(DATA_DIR / "엘지유플러스_요약.csv")
qa = pd.read_csv(DATA_DIR / "엘지유플러스_질의응답.csv")
tables = {"분류": classification, "요약": summarization, "질의응답": qa}

for name, table in tables.items():
    print(f"\n[{name}] 원본 {table.shape[0]:,}행 × {table.shape[1]}열")
    print(pd.DataFrame({"자료형": table.dtypes.astype(str),
                        "결측치": table.isna().sum()}).to_string())
    print("고유 상담 ID 수:", table["source_id"].nunique())
    print("빈 본문 수:", table["input"].str.strip().eq("").sum())
    print("빈 지시 수:", table["instruction"].str.strip().eq("").sum())
    print("빈 출력 수:", table["output"].str.strip().eq("").sum())
    print("작업 유형별 행 수:")
    print(table["task_category"].value_counts().to_string())
    print("행별 본문 글자 수(토큰 수가 아님):")
    print(table["input"].str.len().agg(["min", "mean", "max"]).round(2).to_string())
    table["record_id"] = [f"{name}-{i}" for i in range(len(table))]

practice_ids = [40003, 40005]
for source_id in practice_ids:
    texts = []
    print(f"\n[연습 상담 {source_id}]")
    for name, table in tables.items():
        part = table[table["source_id"] == source_id]
        print(name, "행 수:", len(part))
        texts.extend(part["input"].tolist())
    print("세 파일에서 이 상담의 원문이 모두 동일:", pd.Series(texts).nunique() == 1)
    print("이 상담 원문의 글자 수:", len(texts[0]))

def get_record(table, source_id, task_category, instruction=None):
    selected = table[(table["source_id"] == source_id)
                     & (table["task_category"] == task_category)]
    if instruction is not None:
        selected = selected[selected["instruction"] == instruction]
    if len(selected) != 1:
        raise ValueError("한 행을 선택해야 합니다. ID·작업 유형·질문을 다시 확인하세요.")
    return selected.iloc[0]

def show_record(row, include_input=False):
    print("\nrecord_id:", row["record_id"], " / source_id:", row["source_id"])
    print("작업 유형:", row["task_category"])
    print("지시 또는 질문:", row["instruction"])
    if include_input:
        print("상담 원문:\n" + row["input"])
    print("CSV에 저장된 출력:", row["output"])



[분류] 원본 16,080행 × 6열
                       자료형  결측치
source_id            int64    0
consulting_category    str    0
task_category          str    0
instruction            str    0
input                  str    0
output                 str    0
고유 상담 ID 수: 3216
빈 본문 수: 0
빈 지시 수: 0
빈 출력 수: 0
작업 유형별 행 수:
task_category
상담 요건    3216
상담 주제    3216
상담 사유    3216
상담 내용    3216
상담 결과    3216
행별 본문 글자 수(토큰 수가 아님):
min      670.00
mean    1583.42
max     2836.00

[요약] 원본 11,690행 × 6열
                       자료형  결측치
source_id            int64    0
consulting_category    str    0
task_category          str    0
instruction            str    0
input                  str    0
output                 str    0
고유 상담 ID 수: 3216
빈 본문 수: 0
빈 지시 수: 0
빈 출력 수: 0
작업 유형별 행 수:
task_category
기본 요약          3217
길이 제한 요약       2928
주제 중점 요약       2862
핵심 정보 추출 요약    2683
행별 본문 글자 수(토큰 수가 아님):
min      215.00
mean    1576.97
max     2836.00

[질의응답] 원본 7,497행 × 6열
                       자료형  결측치
source_id        

### 평가 데이터와 기준 준비 — 제공 코드

이번 **상담 요건** 평가 기준은 다음과 같습니다.

- 단일 요건 민원: 중심 요청·주제가 하나입니다. 그 요청의 절차·서류·처리 시점을 여러 번 질문해도 하나의 중심 요청이면 단일로 봅니다.
- 다수 요건 민원: 서로 구분되는 요청·주제가 둘 이상입니다. 상담 중 추가로 이어진 별도 주제도 포함합니다.

평가 상담은 40001·40003·40005·40006·40007입니다. 아래 정답은 이 기준으로 원문을 검토한 **실습용 평가 정답**이며, 데이터 전체에 대한 새 라벨을 만드는 작업은 아닙니다. 원본 CSV의 분류도 별도 컬럼으로 남깁니다.

| 상담 | 평가 정답 | 원문에서 확인할 근거 |
|---|---|---|
| 40001 | 다수 요건 민원 | 케이스·보험 코드·카드 할인·정지 횟수 문의 |
| 40003 | 다수 요건 민원 | 요금제 변경, 결합 할인, 멤버십 변경 관련 상담 |
| 40005 | 다수 요건 민원 | 통화 중 대기 문제와 추가 결합 할인 상담 |
| 40006 | 단일 요건 민원 | 휴대폰 해지라는 한 요청의 절차·서류·시점 문의 |
| 40007 | 다수 요건 민원 | 파손보험 가입·결합 할인·자동이체 카드 확인 |

제공 코드는 선택한 분류·요약 행의 원문 일치를 확인한 뒤 eval_documents를 만듭니다. `show_eval_document(ID)`로 원문을 읽을 수 있습니다. 전체 5개 중 다수 4개·단일 1개이므로 다양한 실무 상황을 대표하는 성능 평가셋이라고 보기는 어렵습니다.


In [10]:
EVAL_IDS = [40001, 40003, 40005, 40006, 40007]
evaluation_labels = {40001: "다수 요건 민원", 40003: "다수 요건 민원", 40005: "다수 요건 민원",
                     40006: "단일 요건 민원", 40007: "다수 요건 민원"}
evaluation_reasons = {
    40001: "케이스·보험 코드·카드 할인·정지 횟수",
    40003: "요금제 변경·결합 할인·멤버십",
    40005: "통화 중 대기와 추가 결합 할인",
    40006: "휴대폰 해지 한 요청의 절차·서류·시점",
    40007: "파손보험 가입·결합 할인·자동이체 카드"
}
eval_rows = []
for sid in EVAL_IDS:
    cr = get_record(classification, sid, "상담 요건")
    sr = get_record(summarization, sid, "길이 제한 요약")
    if cr["input"] != sr["input"]:
        raise ValueError("분류와 요약의 원문을 확인하세요.")
    eval_rows.append({"document_id": sid, "text": cr["input"],
                      "class_record_id": cr["record_id"], "summary_record_id": sr["record_id"],
                      "CSV_분류": cr["output"], "CSV_요약": sr["output"],
                      "정답": evaluation_labels[sid], "정답_근거": evaluation_reasons[sid]})
eval_documents = pd.DataFrame(eval_rows)
print(eval_documents[["document_id", "CSV_분류", "정답", "정답_근거"]].to_string(index=False))
print("\n평가 범주 분포")
print(eval_documents["정답"].value_counts().to_string())

def show_eval_document(sid):
    selected = eval_documents[eval_documents["document_id"] == sid]
    if len(selected) != 1:
        raise ValueError("한 문서를 선택해야 합니다.")
    row = selected.iloc[0]
    print("\n상담 ID:", sid)
    print(row["text"])
    print("기존 요약:", row["CSV_요약"])


 document_id   CSV_분류       정답                 정답_근거
       40001 다수 요건 민원 다수 요건 민원 케이스·보험 코드·카드 할인·정지 횟수
       40003 다수 요건 민원 다수 요건 민원      요금제 변경·결합 할인·멤버십
       40005 다수 요건 민원 다수 요건 민원     통화 중 대기와 추가 결합 할인
       40006 단일 요건 민원 단일 요건 민원 휴대폰 해지 한 요청의 절차·서류·시점
       40007 다수 요건 민원 다수 요건 민원 파손보험 가입·결합 할인·자동이체 카드

평가 범주 분포
정답
다수 요건 민원    4
단일 요건 민원    1


---
## 필수 1. 분류 정확성과 반복 결과의 일관성

### 문제 1-1. 정답 비교와 반복 결과 비교를 나누어 평가하기

#### 문제 설명

평가 상담 5개를 같은 입력·프롬프트·모델·설정으로 세 번 분석했다고 가정한 자료입니다. 이 조건은 **가상 실험의 가정**이며 실제 모델을 실행하지 않습니다.

| 상담 ID | 1회차 | 2회차 | 3회차 |
|---|---|---|---|
| 40001 | 다수 요건 민원 | 다수 요건 민원 | 다수 요건 민원 |
| 40003 | 단일 요건 민원 | 단일 요건 민원 | 단일 요건 민원 |
| 40005 | 다수 요건 민원 | 단일 요건 민원 | 다수 요건 민원 |
| 40006 | 단일 요건 민원 | 단일 요건 민원 | 단일 요건 민원 |
| 40007 | 다수 요건 민원 | 다수 요건 민원 | 다수 요건 민원 |

분류 정확도는 **정답과 일치한 문서 수 ÷ 전체 평가 문서 수**로 계산합니다. 일관성은 이번에 관찰한 세 값이 모두 같은지 확인합니다. 별도 복잡한 일관성 지표는 만들지 않습니다.

#### 요구사항

1. show_eval_document()로 다섯 원문과 정답 근거를 확인하세요.
2. 제공된 repeat_predictions를 document_id로 eval_documents에 연결하여 repeat_results를 만드세요.
3. 각 회차의 정답 여부를 비교하고 평가 문서 수·정답 건수·Accuracy·백분율을 담은 accuracy_table을 만드세요. 분모는 매 회차 5입니다.
4. 각 문서에 `3회_동일`과 `3회_모두정답`을 추가하세요. 전자는 세 출력끼리 비교하고, 후자는 각 출력을 정답과 비교합니다.
5. 한 번이라도 오답인 문서를 wrong_cases로 모으고, 왜 오답인지 원문 기준으로 적으세요.

**보조 구문:** `표.merge(..., on="document_id", validate="one_to_one")`, `표[회차] == 표["정답"]`, `참거짓.sum()`, `&`(두 조건 모두), `표[조건]`

#### 해석 질문

**Q1.** 회차별 정확도와 정답 건수는 얼마인가요? 3회 결과 15개를 한꺼번에 세는 것과 회차별 평가가 같은 의미인가요?

**Q2.** 40003과 40005는 정확성·일관성 측면에서 각각 어떤 사례인가요?

**Q3.** 세 번 같은 결과이면 앞으로도 항상 같거나 반드시 정답이라고 말할 수 있나요?

**Q4.** 전체 정확도 외에 오답 문서와 범주 분포를 확인해야 하는 이유는 무엇인가요? 이 5개만으로 실제 성능을 일반화할 수 있나요?

#### 제출 결과

- repeat_results, accuracy_table, wrong_cases
- 원문에 근거한 오답 설명과 Q1~Q4 답변


### 제공 자료

아래 셀을 실행한 뒤 풀이 코드를 작성하세요.


In [11]:
# 제공 자료: 실제 반복 실행 로그가 아닌 가상 결과입니다.
repeat_predictions = pd.DataFrame({
    "document_id": [40001, 40003, 40005, 40006, 40007],
    "1회차": ["다수 요건 민원", "단일 요건 민원", "다수 요건 민원", "단일 요건 민원", "다수 요건 민원"],
    "2회차": ["다수 요건 민원", "단일 요건 민원", "단일 요건 민원", "단일 요건 민원", "다수 요건 민원"],
    "3회차": ["다수 요건 민원", "단일 요건 민원", "다수 요건 민원", "단일 요건 민원", "다수 요건 민원"]
})


In [12]:
# 문제 1 풀이 코드를 작성하세요.

# 1. 다섯 원문과 정답 근거 확인
for sid in EVAL_IDS:
    show_eval_document(sid)
    print("평가 정답:", evaluation_labels[sid], "/ 근거:", evaluation_reasons[sid])

# 2. 가상 반복 결과를 document_id로 연결
repeat_results = eval_documents[["document_id", "text", "CSV_분류", "정답", "정답_근거"]].merge(
    repeat_predictions, on="document_id", validate="one_to_one")
rounds = ["1회차", "2회차", "3회차"]

# 3. 회차별 정답 여부와 정확도
for r in rounds:
    repeat_results[r + "_정답여부"] = repeat_results[r] == repeat_results["정답"]

acc_rows = []
for r in rounds:
    n_docs = len(repeat_results)
    n_correct = int(repeat_results[r + "_정답여부"].sum())
    acc = n_correct / n_docs
    acc_rows.append({"회차": r, "평가문서수": n_docs, "정답건수": n_correct,
                     "Accuracy": acc, "백분율": f"{acc * 100:.0f}%"})
accuracy_table = pd.DataFrame(acc_rows)
print("\n[accuracy_table]")
print(accuracy_table.to_string(index=False))

# 참고: 15개 결과를 한꺼번에 센 경우 (회차별 평가와 의미가 다름)
total_correct = int(sum(repeat_results[r + "_정답여부"].sum() for r in rounds))
print("15개 결과 전체 정답 수:", total_correct, "/ 15")

# 4. 일관성(세 출력끼리 비교)과 정확성(각 출력과 정답 비교)
repeat_results["3회_동일"] = ((repeat_results["1회차"] == repeat_results["2회차"])
                              & (repeat_results["2회차"] == repeat_results["3회차"]))
repeat_results["3회_모두정답"] = (repeat_results["1회차_정답여부"]
                                & repeat_results["2회차_정답여부"]
                                & repeat_results["3회차_정답여부"])
print("\n[repeat_results] (원문 열 제외)")
print(repeat_results.drop(columns=["text"]).to_string(index=False))

# 5. 한 번이라도 오답인 문서와 원문 기준 설명
wrong_cases = repeat_results[repeat_results["3회_모두정답"] == False].copy()
wrong_cases["오답_회차"] = wrong_cases.apply(
    lambda row: ", ".join(r for r in rounds if not row[r + "_정답여부"]), axis=1)
wrong_reason = {
    40003: "5G 슬림+ 요금제 변경 외에 결합 할인 유지 확인('기존대로 들어간다는 거죠?')과 멤버십 등급이 VIP에서 일반 플러스로 바뀌는 안내가 이어져 주제가 여럿인데, 세 번 모두 단일로 판단함",
    40005: "통화 중 대기 서비스 문제(삭제 후 재신청, 시험 통화)와 별도의 추가 결합 할인 안내가 있어 다수가 맞는데, 2회차에서 뒤쪽 할인 상담을 놓치고 단일로 판단함",
}
wrong_cases["원문 기준 오답 설명"] = wrong_cases["document_id"].map(wrong_reason)
print("\n[wrong_cases]")
print(wrong_cases[["document_id", "정답", "1회차", "2회차", "3회차", "3회_동일",
                   "오답_회차", "원문 기준 오답 설명"]].to_string(index=False))

# 원문 근거 확인
t40003 = eval_documents.loc[eval_documents["document_id"] == 40003, "text"].iloc[0]
t40005 = eval_documents.loc[eval_documents["document_id"] == 40005, "text"].iloc[0]
print("\n40003 원문에 '기존대로 들어간다는 거죠?' 존재:", "기존대로 들어간다는 거죠?" in t40003)
print("40003 원문에 '일반 플러스 등급' 존재:", "일반 플러스 등급" in t40003)
print("40005 원문에 '결합 할인을 더 추가적으로' 존재:", "결합 할인을 더 추가적으로" in t40005)



상담 ID: 40001
상담사: 일상에 즐거운 변화 U+ <NAME>입니다.
고객: 네, 어, 제가 이제 플립3 사전예약 해서 그 케이스 신청했었는데 아직 안 오고 있거든요. 그거는 여기다 문의하는 거 아니죠?
상담사: 네, 어, 고객님, 죄송하지만 지금 상황 제가 여쭙고 케이스 관련해서 문의주신 거 확인해 드리려고 하는데 <NAME> 고객님 명의자분 본인 맞으십니까?
고객: 네, 세 가지를 여쭤볼 거예요. 한 가지가 그 케이스예요.
상담사: 아, 혹시 고객님 죄송하지만 그 대리점 쪽에서 하셨는지 아니면 직영점 쪽에서 하셨는지 아니면 저희 통화 요청하셔 가지고 저희 온라인 샵인 유샵 통해서 하셨는지요?
고객: 네, 음, 저기 직영점이요. 직영점인가, 거기가? 잘 모르겠네요.
상담사: 아, 직영점이요.
고객: 대리점은 아니에요. 아무튼 하나로 마트 옆에 있는 작은 샵이에요.
상담사: 아 그러세요? 아, 그러시면 그 가게에서 하신 거예요?
고객: 그렇죠, 예.
상담사: 아, 그러면 그 사은품 관련해서는 가게에서만 드리는 사은품이 있을 수 있어서 그거는 사은품 가게에도 한번 문의를 주셔야 될 거 같고 어, 다른 문의 사항 있으신 거는 말씀주시겠어요?
고객: 음, 네, 근데 이게 저기 케이스를 제가 온라인으로 주문을 했고요, 그거는. 네, 또 하나가 이제 사전 예약할 수 있는 삼성 케어 플러스라는 보험 상품이 있는데 지금 상품을 이제 제가 신청을 하는 거거든요. 신청을 했어요. 근데 매장 코드를 입력하라는 게 나왔는데, 매장 코드 입력하라는데 매장 코드가 뭔지 몰라서 저희가 이제 구입한 데 가서 매장 코드가 뭐냐고 물어봤더니 고객센터에 전화를 해보라는 거죠.
상담사: 아, 네.
고객: 자기들은 이 매장 코드는 모른다는 거예요.
상담사: 고객님, 네.
고객: 예, 그래서 지금 가입하는 날짜가 얼마 안 남았어요. 그래 가지고 어떻게 해야 되나 싶어서 전화 드렸어요.
상담사: 아, 고객님, 근데 저희 고객센터에서 개통을 해드리는 게 아니어서 이거는 매장 쪽으로 문의주시는

### 해석 답안 작성

- Q1. 1회차는 5개 중 4개 정답으로 Accuracy 0.8(80%), 2회차는 3개 정답으로 0.6(60%), 3회차는 4개 정답으로 0.8(80%)이다. 15개 결과를 한꺼번에 세면 11개 정답으로 약 73% 정도가 나오는데, 이건 회차별 평가와 의미가 다르다. 회차별 평가는 "한 번 실행했을 때 문서 5개를 얼마나 맞히는가"를 보는 것이고, 2회차에서 정확도가 0.6으로 떨어진 것처럼 실행마다 결과가 달라지는 걸 보여 준다. 15개를 합치면 이런 회차 간 차이가 평균으로 섞여서 안 보이고, 같은 문서가 세 번씩 들어가니 평가 문서가 15개인 것처럼 착각할 수도 있다.
-> 1, 3회차는 80%, 2ㅎ회차는 60%
-> 15개를 한꺼번에 세면 반복 관측 전체에 대한 요약이 되어 회차별 변동이 보이지 않을 수 있다.
-> 이번 문제는 같은 5개의 문서의 각 회차 정확도를ㄹ 구하는 것으로 분모를 매법 5로 둔다.

- Q2. 40003은 세 번 모두 "단일 요건 민원"으로 같은 결과가 나와서 일관성은 있지만(3회_동일 True) 세 번 다 틀렸다(3회_모두정답 False). 원문에서 요금제 변경 외에 결합 할인 유지 확인("기존대로 들어간다는 거죠?")과 멤버십 등급이 일반 플러스로 바뀌는 안내까지 이어져서 다수 요건이 맞기 때문이다. 즉 일관되게 틀리는 사례다. 40005는 1회차와 3회차는 정답(다수)인데 2회차만 단일로 틀려서 일관성이 없는 사례다(3회_동일 False). 통화 중 대기 문제와 추가 결합 할인 안내라는 두 주제 중 뒤쪽 할인 상담을 2회차에서 놓친 것으로 볼 수 있다.
-> 40003은 출력이 안정적으로 같지만 세 번 모두 틀렸다. 40005는 두 번 맞고 한 번 틀리며 출력도 변했다.
-> 정확성은 정답과 비교하고 일관성은 반복 결과끼리 비교한다는 차이를 보여준다.

- Q3. 말할 수 없다. 세 번 같았다는 건 이번에 관찰한 세 번의 결과가 같았다는 것뿐이라서, 다음에 실행하면 다른 결과가 나올 가능성을 없애 주지 않는다. 또 40003처럼 세 번 모두 같아도 모두 틀릴 수 있다. 일관성은 결과끼리 같은지를 보는 것이고 정확성은 정답과 맞는지를 보는 것이라서 서로 다른 기준이다. 그리고 이번 자료는 가상 실험의 가정이라 실제 모델의 반복 결과도 아니다.
-> 그렇지 않다. 세 번 관찰한 범위에서 같았다고 할 수는 있지만 이후 모든 실행을 보장하지 않는다.

- Q4. 전체 정확도 0.8만 보면 어떤 문서를 틀렸는지, 왜 틀렸는지 알 수 없다. 오답 문서를 봐야 40003처럼 계속 틀리는 유형인지, 40005처럼 가끔 흔들리는 유형인지 구분해서 개선 방향을 정할 수 있다. 범주 분포도 확인해야 하는데, 이번 평가셋은 다수 4개, 단일 1개라서 모든 문서를 "다수"로 답해도 정확도가 0.8이 나온다. 그래서 단일 요건을 잘 구분하는지는 이 숫자로 알 수 없다. 5개는 너무 적고 범주도 한쪽으로 쏠려 있어서 이 결과로 실제 성능을 일반화할 수 없다.
-> 특정 문서나 범주에 오류가 몰릴 수 있기 때문에 전체 비율만으로 문제를 알기는 어렵다.
-> 오답 사례를 포함해 직접 확인하는 과정이 필요하다.


---
## 필수 2. 정보 추출과 요약에 맞는 평가 기준 적용

### 문제 2-1. 표현 차이·잘못된 값·누락·원문 밖 정보 구분

#### 문제 설명

**정보 추출:** 상담 40003의 질의응답 3개를 이용해 결합 할인 수혜자, 초과 사용분 과금 여부, 기존 결합 할인 유지 여부를 확인합니다. 가상 결과는 각각 `남편분`, `추가 요금이 발생함`, 빈 문자열입니다.

먼저 평가 기준을 정합니다. 필수 항목에 비어 있지 않은 답이 있어야 하고, 같은 의미의 표현은 맞는 답으로 인정합니다. `남편`과 `남편분`은 이번 질문에서 같은 대상을 가리키므로 의미상 일치로 봅니다. 코드의 문자열 비교와 사람이 판단하는 의미 비교를 구분하세요.

**요약:** 상담 40005의 세 가상 요약을 평가합니다. 핵심 정보는 ① 통화 중 대기 서비스 재설정과 정상 확인, ② 추가 할인은 즉시 진행하지 않고 추후 상담 연락에 동의한 점입니다. 다음 세 기준을 각각 판단합니다.

| 기준 | True의 뜻 |
|---|---|
| 사실_일치 | 요약에 적힌 사실이 원문과 일치함 |
| 핵심_유지 | 위에서 지정한 핵심 정보 두 가지를 모두 유지함 |
| 원문밖_정보 | 원문에 없는 내용을 추가함. 이 열은 True이면 문제 있음 |

핵심 누락과 사실 왜곡은 구분합니다. 어떤 사실을 쓰지 않은 경우는 핵심_유지에서, 반대로 바꾸어 쓴 경우는 사실_일치에서 확인합니다. 한 요약에 여러 오류가 함께 있을 수 있습니다.

#### 요구사항

1. 제공 셀을 실행하여 질문·기존 답변·가상 추출 결과와 요약 A~C를 확인하세요. 원문은 필수 1에서 확인했으며 필요하면 다시 출력합니다.
2. 추출 결과에 `문자열_일치`, `항목_존재`, `의미_일치`, `판단_근거`를 추가하여 extraction_review를 만드세요. 앞 두 항목은 코드로 계산하고, 뒤 두 항목은 사람이 원문을 읽고 입력합니다.
3. 추출 결과의 정확한 필드 수와 전체 필드 수를 확인하세요. 표현 차이를 모두 오답으로 세거나 빈 값을 부정 답변으로 해석하지 않습니다.
4. 요약마다 세 기준과 근거를 적은 summary_review를 만드세요. CSV 요약과 문장이 완전히 같아야 한다는 기준을 사용하지 않습니다.
5. 발견한 정보 추출 오류·누락·요약 오류·환각을 구분하고 각 문제에 맞는 개선 방향을 적으세요.

#### 해석 질문

**Q1.** 문자열이 다르면 항상 추출 오답인가요? 남편분과 빈 문자열을 각각 어떻게 평가해야 하나요?

**Q2.** 요약 A와 C는 어떤 점에서 다르며, 요약 B에는 어떤 오류가 함께 있나요?

**Q3.** 자연스러운 문장이나 CSV 요약과의 문자열 일치만으로 요약 품질을 판단하면 안 되는 이유는 무엇인가요?

**Q4.** 원문에 없는 원인 생성, 핵심 누락, 출력 형식 오류에는 각각 어떤 개선이 필요한가요? 스키마만 수정하면 내용 오류도 해결되나요?

#### 제출 결과

- extraction_review, summary_review
- 필드별·요약별 판단 근거와 개선 방향
- Q1~Q4 답변


### 제공 자료

아래 셀을 실행한 뒤 풀이 코드를 작성하세요.


In [13]:
extraction_specs = [
    ("결합 할인 수혜자", "의문사형", "고객의 결합 할인은 누구 앞으로 되어 있나요?", "남편분"),
    ("초과 사용분 과금", "의문사형", "요금제 변경 시 초과 사용분은 어떻게 처리되나요?", "추가 요금이 발생함"),
    ("기존 결합 할인 유지", "예/아니요형", "고객의 요금제 변경 시 기존 결합 할인은 유지되나요?", ""),
]
extraction_rows=[]
for field, task, question, predicted in extraction_specs:
    qr=get_record(qa,40003,task,question)
    extraction_rows.append({"source_id":40003,"record_id":qr["record_id"],"필드":field,
                           "질문":qr["instruction"],"원문":qr["input"],"기존_answer":qr["output"],
                           "가상_추출":predicted})
extraction_data=pd.DataFrame(extraction_rows)
print(extraction_data[["필드","질문","기존_answer","가상_추출"]].to_string(index=False))

# 작성자가 만든 가상 요약이며, 일부는 의도적으로 부정확합니다.
summary_candidates=pd.DataFrame({
    "예시":["A","B","C"],
    "source_id":[40005,40005,40005],
    "가상_요약":[
        "통화 중 대기 서비스를 재설정하고 시험 통화로 정상 작동을 확인했다. 추가 결합 할인은 즉시 연결하지 않고 추후 상담 연락을 받기로 했다.",
        "단말기 고장으로 통화 중 대기가 작동하지 않아 고객이 새 휴대폰을 구매했다. 고객은 추가 결합 할인 가입도 즉시 완료했다.",
        "통화 중 대기 서비스를 재설정한 후 시험 통화로 정상 작동을 확인했다."
    ]
})
print("\n기존 요약:", get_record(summarization,40005,"길이 제한 요약")["output"])
print(summary_candidates.to_string(index=False))


         필드                            질문            기존_answer      가상_추출
  결합 할인 수혜자     고객의 결합 할인은 누구 앞으로 되어 있나요?                   남편        남편분
  초과 사용분 과금   요금제 변경 시 초과 사용분은 어떻게 처리되나요?              과금되지 않음 추가 요금이 발생함
기존 결합 할인 유지 고객의 요금제 변경 시 기존 결합 할인은 유지되나요? 예, 결합 할인은 그대로 유지됩니다.           

기존 요약: 고객은 통화 중 대기 서비스의 작동 방식에 대해 문의하였고, 상담사는 서비스 삭제 후 재설정하여 정상 작동을 확인해주었다. 추가로 LGU+ 결합 할인 혜택에 대해 안내하였으나 고객은 추후 검토하기로 하였다.
예시  source_id                                                                         가상_요약
 A      40005 통화 중 대기 서비스를 재설정하고 시험 통화로 정상 작동을 확인했다. 추가 결합 할인은 즉시 연결하지 않고 추후 상담 연락을 받기로 했다.
 B      40005          단말기 고장으로 통화 중 대기가 작동하지 않아 고객이 새 휴대폰을 구매했다. 고객은 추가 결합 할인 가입도 즉시 완료했다.
 C      40005                                       통화 중 대기 서비스를 재설정한 후 시험 통화로 정상 작동을 확인했다.


In [14]:
# 문제 2 풀이 코드를 작성하세요.

# 1. 원문 다시 확인이 필요하면 출력 (40003 근거 문장만 확인)
text_40003 = extraction_data["원문"].iloc[0]
for p in ["저희 남편 앞으로 되어 있나요?", "초과 사용분이 따로 있진 않아서 과금되진 않으세요",
          "따로 할인을 유지되실 거예요"]:
    print(f"40003 원문에 '{p}' 존재:", p in text_40003)

# 2. 정보 추출 검토 (앞 두 열은 코드 계산, 뒤 두 열은 사람이 원문을 읽고 입력)
extraction_review = extraction_data.drop(columns=["원문"]).copy()
extraction_review["문자열_일치"] = extraction_review["가상_추출"] == extraction_review["기존_answer"]
extraction_review["항목_존재"] = extraction_review["가상_추출"].str.strip() != ""
extraction_review["의미_일치"] = [True, False, False]
extraction_review["판단_근거"] = [
    "'저희 남편 앞으로 되어 있나요?' '어, 네, 맞습니다.' 남편분은 남편과 같은 대상이라 표현 차이만 있음",
    "원문은 '초과 사용분이 따로 있진 않아서 과금되진 않으세요'로 과금되지 않음. 추출 결과는 반대 의미라 잘못된 값",
    "빈 문자열이라 답이 누락됨. '아니요'로 해석하면 안 되고, 원문상 정답은 유지됨('따로 할인을 유지되실 거예요')",
]
print("\n[extraction_review]")
print(extraction_review[["필드", "기존_answer", "가상_추출", "문자열_일치", "항목_존재",
                         "의미_일치", "판단_근거"]].to_string(index=False))

# 3. 정확한 필드 수
n_fields = len(extraction_review)
print("\n문자열로 일치한 필드 수:", int(extraction_review["문자열_일치"].sum()), "/", n_fields)
print("의미상 정확한 필드 수:", int(extraction_review["의미_일치"].sum()), "/", n_fields)
print("값이 있는 필드 수:", int(extraction_review["항목_존재"].sum()), "/", n_fields)

# 4. 요약 검토 (사람이 원문을 읽고 판단, CSV 요약과의 문장 일치는 기준이 아님)
text_40005 = eval_documents.loc[eval_documents["document_id"] == 40005, "text"].iloc[0]
for w in ["고장", "구매", "다음에 할게요", "네, 그렇게 해주세요"]:
    print(f"40005 원문에 '{w}' 존재:", w in text_40005)

summary_review = summary_candidates.copy()
summary_review["사실_일치"] = [True, False, True]
summary_review["핵심_유지"] = [True, False, False]
summary_review["원문밖_정보"] = [False, True, False]
summary_review["판단_근거"] = [
    "재설정과 시험 통화 확인, 추가 할인은 즉시 연결 안 하고 추후 연락에 동의한 점 모두 원문과 일치",
    "'단말기 고장', '새 휴대폰 구매'는 원문에 없음(환각). 고객은 '다음에 할게요'라고 했는데 할인 가입 즉시 완료로 바꿈(사실 왜곡). 재설정 정상 확인과 추후 연락도 빠짐",
    "적힌 내용은 원문과 맞지만 두 번째 핵심인 추가 할인 보류와 추후 연락 동의가 빠짐(핵심 누락)",
]
summary_review["CSV요약과_문자열일치"] = summary_review["가상_요약"] == get_record(summarization, 40005, "길이 제한 요약")["output"]
print("\n[summary_review]")
print(summary_review.drop(columns=["가상_요약"]).to_string(index=False))

# 5. 문제 유형 구분과 개선 방향
problem_types = pd.DataFrame([
    {"대상": "추출: 초과 사용분 과금", "유형": "정보 추출 오류(잘못된 값)",
     "개선 방향": "상담사가 확인해 준 발화를 근거로 답하게 하고, 근거 문장을 함께 출력하게 해서 검토"},
    {"대상": "추출: 기존 결합 할인 유지", "유형": "누락(빈 값)",
     "개선 방향": "필수 항목은 비우지 말고 원문에 없으면 '확인 불가'로 쓰게 하고, 빈 값은 재검토 대상으로 표시"},
    {"대상": "요약 B: 할인 가입 즉시 완료", "유형": "요약 오류(사실 왜곡)",
     "개선 방향": "실제 완료한 일과 추후 예정 사항을 구분하라는 조건을 분명히 쓰고 원문과 대조"},
    {"대상": "요약 B: 단말기 고장, 새 휴대폰 구매", "유형": "환각(원문 밖 정보)",
     "개선 방향": "원문에 없는 원인이나 사건을 추가하지 말라는 조건과 근거 확인 단계 추가"},
    {"대상": "요약 C: 추가 할인 내용 없음", "유형": "요약 오류(핵심 누락)",
     "개선 방향": "반드시 포함할 핵심 정보 두 가지를 지시에 명시하고 포함 여부 점검"},
])
print("\n[문제 유형과 개선 방향]")
print(problem_types.to_string(index=False))


40003 원문에 '저희 남편 앞으로 되어 있나요?' 존재: True
40003 원문에 '초과 사용분이 따로 있진 않아서 과금되진 않으세요' 존재: True
40003 원문에 '따로 할인을 유지되실 거예요' 존재: True

[extraction_review]
         필드            기존_answer      가상_추출  문자열_일치  항목_존재  의미_일치                                                            판단_근거
  결합 할인 수혜자                   남편        남편분   False   True   True     '저희 남편 앞으로 되어 있나요?' '어, 네, 맞습니다.' 남편분은 남편과 같은 대상이라 표현 차이만 있음
  초과 사용분 과금              과금되지 않음 추가 요금이 발생함   False   True  False  원문은 '초과 사용분이 따로 있진 않아서 과금되진 않으세요'로 과금되지 않음. 추출 결과는 반대 의미라 잘못된 값
기존 결합 할인 유지 예, 결합 할인은 그대로 유지됩니다.              False  False  False 빈 문자열이라 답이 누락됨. '아니요'로 해석하면 안 되고, 원문상 정답은 유지됨('따로 할인을 유지되실 거예요')

문자열로 일치한 필드 수: 0 / 3
의미상 정확한 필드 수: 1 / 3
값이 있는 필드 수: 2 / 3
40005 원문에 '고장' 존재: False
40005 원문에 '구매' 존재: False
40005 원문에 '다음에 할게요' 존재: True
40005 원문에 '네, 그렇게 해주세요' 존재: True

[summary_review]
예시  source_id  사실_일치  핵심_유지  원문밖_정보                                                                                              판단_근거  CSV요

### 해석 답안 작성

- Q1. 항상 오답은 아니다. 코드로 비교하면 세 필드 모두 문자열 일치가 False지만, 의미로 보면 다르다. "남편분"은 "남편"과 문자열은 다르지만 원문에서 고객이 "저희 남편 앞으로 되어 있나요?"라고 묻고 상담사가 "어, 네, 맞습니다"라고 한 그 대상을 가리키니까 표현 차이일 뿐 의미상 정답이다. 빈 문자열은 값이 없는 누락이라서 의미_일치 False로 보되, "아니요"라고 답한 것으로 해석하면 안 된다. 원문은 "따로 할인을 유지되실 거예요"라서 정답은 "유지됨"이고, 빈 값은 재검토 대상으로 남겨야 한다. 참고로 "추가 요금이 발생함"은 원문의 "과금되진 않으세요"와 반대라서 표현 차이가 아니라 잘못된 값이다. 결국 의미상 정확한 필드는 3개 중 1개다.
->  표현이 다르지만 같은 대상을 뜻하면 맞게 인정할 수 있다.
-> 이번에는 남편분을 남편과 의미상 같다고 사전에 정했다.
-> 반 문자열은 답을 얻지 못한 상태이므로 항목 누락에 해당한다.(부정으로 해석할 수 없다.)

- Q2. A와 C는 둘 다 적힌 내용은 원문과 맞아서 사실_일치는 True이고 원문 밖 정보도 없다. 차이는 핵심 유지인데, A는 재설정 후 정상 확인과 추가 할인 보류 및 추후 연락 동의를 모두 담았고, C는 첫 번째 핵심만 있고 추가 할인 부분이 빠져서 핵심_유지가 False다. C는 틀린 말을 한 게 아니라 필요한 말을 안 한 것이다. B에는 오류가 세 가지가 함께 있다. 원문에 없는 "단말기 고장"과 "새 휴대폰 구매"를 만들어 낸 환각(코드로 확인해도 "고장", "구매"는 원문에 없음), 고객이 "다음에 할게요"라고 했는데 할인 가입을 즉시 완료했다고 바꾼 사실 왜곡, 그리고 재설정 후 정상 확인과 추후 연락이 빠진 핵심 누락이다.
-> A는 두 핵심을 모두 유지하지만 C는 서비스 재설정만 담아 할인 상담의 결론을 빠뜨렸다.
-> B에서는 원문 밖 원인, 구매 사실을 생성한 환각(할루시네이션)과 즉시 가입으로 결론을 바꾼 사실 왜곡이 함께 있다.

- Q3. 문장이 자연스러워도 B처럼 내용이 틀릴 수 있다. B는 매끄럽게 읽히지만 원문에 없는 사건과 반대 사실이 들어 있다. 또 CSV 요약과 문자열이 같은지로 판단하면, 실제로 A, B, C 모두 CSV 요약과 문자열 일치가 False라서 정확한 A까지 다 틀린 것이 된다. 같은 내용도 표현이 여러 가지라서 요약은 문장이 같은지가 아니라 원문 사실과 맞는지, 핵심이 빠지지 않았는지, 원문 밖 내용이 없는지로 평가해야 한다. CSV 요약도 원문에 비춰 확인할 참고 자료일 뿐 정답이라고 단정할 수 없다.
-> 같은 사실을 여러 문장으로 표현할 수 있으므로 정답 요약 한 문장과의 완전 일치가 필수는 아니다.
-> 반대로 자연스러운 문장도 B처럼 근거 없는 사실이나 잘못된 결론을 담을 수 있다.
-> 원문의 사실과 정해둔 핵심 정보를 기준으로 읽어야 한다.

- Q4. 원문에 없는 원인을 만드는 경우에는 원문에 있는 사실만 쓰고 원인이 안 나오면 쓰지 말라는 조건을 넣고, 결과의 주장마다 원문 근거를 확인하는 검토 단계를 둔다. 핵심 누락은 반드시 포함할 정보(이번에는 두 가지)를 지시에 명확히 적고, 결과에 다 들어갔는지 점검한다. 출력 형식 오류는 스키마나 출력 형식 조건(필수 필드, 자료형, 허용 값)을 정하고 구조 검사를 하면 된다. 하지만 스키마만 고친다고 내용 오류가 해결되지는 않는다. 스키마는 필드가 있고 문자열인지 같은 모양만 보장해 줄 뿐이라서, "추가 요금이 발생함"처럼 형식은 맞는데 원문과 반대인 값은 여전히 통과한다. 내용은 원문 대조로 따로 확인해야 한다.
-> 원인 확각에서는 원문 근거 사용, 확인 불가 처리 기준을 핵심 누락에는 필수 포함 정보를
-> 출력 형식 오류에는 스키마와 구조 검증을 보와할 수 잇다.
-> 스키마는 필드, 자료형을 제한하지만 그 문자열의 내용이 참인지 보장하지는 않기 때문에
-> 내용 오류는 별도의 개선, 재평가 작업을 수행해야 한다.


---
## 과제 1. 동일한 평가 데이터로 방법 비교와 개선 검증

### 문제 3-1. 개선된 사례·새 오류·남은 오류를 함께 확인하기

#### 문제 설명

같은 평가 상담 5개에 두 프롬프트를 적용했다고 가정합니다.

- **변경 전 A, Zero-shot:** 공통 분류 기준만 제공합니다.
- **변경 후 B, Few-shot:** 같은 기준에 별도 상담 40002·40011의 정답 예시를 추가합니다.

두 예시는 평가 상담 5개와 분리합니다. 원문을 확인하면 40002는 명의변경이라는 한 요청의 절차·서류·변경 시 혜택·방문 안내, 40011은 영화 월정액 해지·요금제 검토·멤버십 신청을 다룹니다. 예시 정답은 각각 단일·다수 요건 민원입니다.

입력·평가 정답·모델·생성 설정·출력 조건은 같고 예시 추가만 달라졌다고 **가정한 비교 자료**입니다. 필수 1의 반복 평가와 별도인 가상 실험이며, 실제 API를 호출한 결과가 아닙니다.

| 상담 | 변경 전 A | 변경 후 B |
|---|---|---|
| 40001 | 다수 요건 민원 | 다수 요건 민원 |
| 40003 | 단일 요건 민원 | 다수 요건 민원 |
| 40005 | 단일 요건 민원 | 다수 요건 민원 |
| 40006 | 단일 요건 민원 | 다수 요건 민원 |
| 40007 | 단일 요건 민원 | 단일 요건 민원 |

추가로 **단일 호출과 순차 처리**가 같은 문서에서 분류·요약을 수행한 별도 가상 결과를 제공합니다. 분류는 정답 비교로, 요약은 원문과의 사실 일치로 평가합니다. 호출 수는 정상적으로 처리되는 설계의 문서당 예상 횟수이며 실제 과금 기록이 아닙니다.

#### 요구사항

1. 제공된 example_table과 eval_documents의 ID 및 원문이 겹치지 않는지 확인하세요. 평가 정답을 예시로 넣지 않습니다.
2. 상담 40006에 대한 zero_prompt와 few_prompt를 구성하세요. 공통 기준·대상 원문은 동일하게 두고, Few-shot에만 두 예시의 원문·정답을 추가합니다. 프롬프트 문자열을 변수에 보관하고 구성 차이를 확인합니다.
3. 제공된 comparison_predictions를 ID로 연결하여 prompt_comparison을 만들고, 두 방법의 정답 건수와 Accuracy를 계산하세요.
4. 각 문서를 `정답 유지`, `개선`, `새 오류`, `오답 유지`로 구분하세요. 두 방법의 전체 정확도만 비교하지 않습니다.
5. 40006의 새 오류와 40007의 남은 오류에 대해 원문 근거, 가능한 원인, 추가 개선안을 작성하세요. 원인은 확정 사실이 아니라 검토할 가설로 적습니다. 작성한 개선안을 실제로 실행한 것처럼 표현하지 않습니다.
6. 제공된 workflow_outputs의 두 방식에 대해 분류 정답 건수·Accuracy, 사람이 판단한 요약 사실 일치 건수, 문서당·전체 예상 호출 수를 workflow_comparison으로 정리하세요. 제공된 요약을 읽어 수동 판단 근거도 확인합니다.
7. 두 비교를 바탕으로 어느 방식을 우선 검토할지 조건을 붙여 설명하고, 반복 실행·다른 데이터·처리 시간과 사용량에서 더 확인할 사항을 적으세요.

**판정 기준**

| 변경 전 정답 | 변경 후 정답 | 변화 |
|---|---|---|
| True | True | 정답 유지 |
| False | True | 개선 |
| True | False | 새 오류 |
| False | False | 오답 유지 |

**보조 구문:** `set(A) & set(B)`는 공통 값을 찾습니다. `예측 == 정답`, `참거짓.sum()`, `정답건수 / 평가문서수`, `pd.DataFrame(행_목록)`을 사용하세요.

#### 해석 질문

**Q1.** 두 프롬프트의 정확도는 얼마이며, 개선·새 오류·오답 유지 문서는 각각 무엇인가요? 변경 후 방식이 항상 좋다고 결론 내릴 수 있나요?

**Q2.** Few-shot 예시와 평가 데이터를 분리하는 이유는 무엇인가요? 서로 다른 문서로 두 방식을 평가하면 어떤 문제가 있나요?

**Q3.** 단일 호출과 순차 처리 비교에서 분류와 요약 결과는 각각 어떻게 달랐나요? 호출이 두 배면 실제 시간·비용도 정확히 두 배인가요?

**Q4.** 추가 개선안을 어떻게 검증해야 하나요? 같은 평가 정답에 맞춰 계속 수정하는 경우와, 출력 누락·중간 분류 오류가 발생한 경우를 함께 설명하세요.

#### 제출 결과

- 예시·평가 분리 확인, 두 프롬프트 문자열
- prompt_comparison 및 정확도·변화 유형 집계
- 오류별 개선안, workflow_comparison, 추가 검증 계획
- Q1~Q4 답변


### 제공 자료

아래 셀을 실행한 뒤 풀이 코드를 작성하세요.


In [15]:
# 제공 자료: 평가셋 밖의 실제 상담을 프롬프트 예시로 준비합니다.
example_rows=[]
for sid, expected, reason in [
    (40002,"단일 요건 민원","명의변경 한 요청의 절차·서류·변경 영향·방문 안내"),
    (40011,"다수 요건 민원","영화 월정액 해지·요금제 검토·멤버십 신청")
]:
    r=get_record(classification,sid,"상담 요건")
    example_rows.append({"document_id":sid,"text":r["input"],"CSV_분류":r["output"],
                         "예시정답":expected,"판단근거":reason})
example_table=pd.DataFrame(example_rows)
BASE_INSTRUCTION=(
    "상담 요건을 단일 요건 민원 또는 다수 요건 민원 중 하나로 분류하세요. "
    "중심 요청·주제가 하나면 단일, 구분되는 요청·주제가 둘 이상이면 다수입니다. "
    "한 요청의 절차·서류·시점을 여러 번 질문한 것만으로 다수로 보지 마세요. "
    "상담 중 추가로 이어진 별도 주제도 포함하여 판단하세요. 범주명만 출력하세요."
)
comparison_predictions=pd.DataFrame({
    "document_id":[40001,40003,40005,40006,40007],
    "Zero-shot":["다수 요건 민원","단일 요건 민원","단일 요건 민원","단일 요건 민원","단일 요건 민원"],
    "Few-shot":["다수 요건 민원","다수 요건 민원","다수 요건 민원","다수 요건 민원","단일 요건 민원"]
})

# 별도 가상 워크플로우 비교입니다. 같은 입력·정답·평가 기준을 사용한다고 가정합니다.
# 두 방식의 요약은 비교를 단순화하려고 같은 문장으로 제공했습니다.
workflow_outputs=pd.DataFrame({
    "document_id":[40001,40003,40005,40006,40007],
    "단일_분류":["다수 요건 민원","다수 요건 민원","다수 요건 민원","다수 요건 민원","단일 요건 민원"],
    "순차_분류":["다수 요건 민원","다수 요건 민원","다수 요건 민원","단일 요건 민원","단일 요건 민원"],
    "공통_가상요약":[
        "케이스, 보험 코드, 카드 할인과 휴대폰 정지 횟수에 관한 문의와 안내가 있었다.",
        "요금제 변경 후 기존 결합 할인은 해지되었고 VIP 등급은 그대로 유지되었다.",
        "통화 중 대기 서비스를 재설정해 정상 작동을 확인했으며 추가 할인은 추후 상담 연락을 받기로 했다.",
        "휴대폰 해지에 필요한 서류와 팩스 또는 매장 방문 절차, 처리 시점에 관한 안내가 있었다.",
        "파손보험 가입과 결합 할인을 위한 방문 안내를 받고 등록된 자동이체 카드를 확인했다."
    ],
    # 사람이 원문과 비교한 판정입니다. 자동 평가 모델의 출력이 아닙니다.
    "요약_사실일치":[True,False,True,True,True]
})
print("Few-shot 예시")
print(example_table[["document_id","예시정답","판단근거"]].to_string(index=False))
print("\n가상 워크플로우 요약과 사람의 사실 일치 판정")
print(workflow_outputs[["document_id","공통_가상요약","요약_사실일치"]].to_string(index=False))


Few-shot 예시
 document_id     예시정답                         판단근거
       40002 단일 요건 민원 명의변경 한 요청의 절차·서류·변경 영향·방문 안내
       40011 다수 요건 민원      영화 월정액 해지·요금제 검토·멤버십 신청

가상 워크플로우 요약과 사람의 사실 일치 판정
 document_id                                                 공통_가상요약  요약_사실일치
       40001           케이스, 보험 코드, 카드 할인과 휴대폰 정지 횟수에 관한 문의와 안내가 있었다.     True
       40003             요금제 변경 후 기존 결합 할인은 해지되었고 VIP 등급은 그대로 유지되었다.    False
       40005 통화 중 대기 서비스를 재설정해 정상 작동을 확인했으며 추가 할인은 추후 상담 연락을 받기로 했다.     True
       40006      휴대폰 해지에 필요한 서류와 팩스 또는 매장 방문 절차, 처리 시점에 관한 안내가 있었다.     True
       40007         파손보험 가입과 결합 할인을 위한 방문 안내를 받고 등록된 자동이체 카드를 확인했다.     True


In [16]:
# 문제 3 풀이 코드를 작성하세요.

# 1. Few-shot 예시와 평가 데이터 분리 확인
eval_id_set = set(eval_documents["document_id"])
example_id_set = set(example_table["document_id"])
print("겹치는 ID:", eval_id_set & example_id_set)
print("겹치는 원문 수:", len(set(eval_documents["text"]) & set(example_table["text"])))
print("예시 정답 확인:")
print(example_table[["document_id", "CSV_분류", "예시정답", "판단근거"]].to_string(index=False))

# 2. 상담 40006에 대한 두 프롬프트 (공통 기준과 대상 원문은 동일)
target = eval_documents[eval_documents["document_id"] == 40006].iloc[0]
target_block = "[분류할 상담]\n상담 원문:\n" + target["text"] + "\n분류:"

zero_prompt = BASE_INSTRUCTION + "\n\n" + target_block

example_block = ""
for i, ex in enumerate(example_table.itertuples(index=False), start=1):
    example_block += f"[예시 {i}]\n상담 원문:\n" + ex.text + "\n분류: " + ex.예시정답 + "\n\n"
few_prompt = BASE_INSTRUCTION + "\n\n" + example_block + target_block

print("\n[zero_prompt 앞부분]")
print(zero_prompt[:260] + " ...")
print("\n[few_prompt 구성 확인]")
print("두 프롬프트 모두 공통 기준으로 시작:", zero_prompt.startswith(BASE_INSTRUCTION) and few_prompt.startswith(BASE_INSTRUCTION))
print("두 프롬프트 모두 같은 대상 블록으로 끝남:", zero_prompt.endswith(target_block) and few_prompt.endswith(target_block))
print("zero_prompt에 예시 원문 없음:", all(t not in zero_prompt for t in example_table["text"]))
print("few_prompt에 예시 원문 두 개 포함:", all(t in few_prompt for t in example_table["text"]))
print("few_prompt의 예시 수:", few_prompt.count("[예시 "))
print("글자 수 zero / few:", len(zero_prompt), "/", len(few_prompt))
print("늘어난 부분이 예시 블록과 같음:", few_prompt.replace(example_block, "") == zero_prompt)

# 3. 정답 비교와 정확도
prompt_comparison = eval_documents[["document_id", "정답"]].merge(
    comparison_predictions, on="document_id", validate="one_to_one")
prompt_comparison["A_정답"] = prompt_comparison["Zero-shot"] == prompt_comparison["정답"]
prompt_comparison["B_정답"] = prompt_comparison["Few-shot"] == prompt_comparison["정답"]
n_eval = len(prompt_comparison)
acc_A = prompt_comparison["A_정답"].sum() / n_eval
acc_B = prompt_comparison["B_정답"].sum() / n_eval
print("\nZero-shot 정답 건수:", int(prompt_comparison["A_정답"].sum()), "/", n_eval, "/ Accuracy:", acc_A)
print("Few-shot 정답 건수:", int(prompt_comparison["B_정답"].sum()), "/", n_eval, "/ Accuracy:", acc_B)

# 4. 문서별 변화 유형
def change_type(before, after):
    if before and after:
        return "정답 유지"
    elif not before and after:
        return "개선"
    elif before and not after:
        return "새 오류"
    else:
        return "오답 유지"

prompt_comparison["변화"] = [change_type(a, b) for a, b in
                            zip(prompt_comparison["A_정답"], prompt_comparison["B_정답"])]
print("\n[prompt_comparison]")
print(prompt_comparison.to_string(index=False))
print("\n변화 유형별 건수:")
print(prompt_comparison["변화"].value_counts().to_string())

# 5. 새 오류와 남은 오류 분석 (원인은 검토할 가설, 개선안은 아직 실행하지 않음)
error_analysis = pd.DataFrame([
    {"document_id": 40006, "변화": "새 오류",
     "원문 근거": "해지 서류, 팩스, 대리점 방문, 처리 시간 질문이 모두 '휴대폰 해지' 한 요청의 절차와 시점이라 단일",
     "가능한 원인(가설)": "다수 예시(40011)에 '해지'가 들어 있어 해지라는 단어에 끌렸거나, 질문 횟수가 많은 점을 여러 주제로 봤을 가능성",
     "추가 개선안(미실행)": "평가셋 밖의 '한 요청 절차를 여러 번 묻는 해지 상담'을 단일 예시로 추가하고, 단어가 아니라 요청 수로 판단하라는 문장을 보강"},
    {"document_id": 40007, "변화": "오답 유지",
     "원문 근거": "파손보험 가입, 결합 할인 서류 문의, 상담 끝 '하나만 더'로 이어진 자동이체 카드 확인까지 세 주제라 다수",
     "가능한 원인(가설)": "보험과 결합 할인이 모두 '대리점 방문'으로 안내되어 하나의 방문 안내로 묶어 봤거나, 뒤에 추가된 카드 확인을 놓쳤을 가능성",
     "추가 개선안(미실행)": "방문 장소가 같아도 주제가 다르면 각각 센다는 기준과 '상담 후반에 추가된 질문도 포함' 조건을 강조, 비슷한 유형의 평가셋 밖 예시 추가"},
])
print("\n[error_analysis]")
print(error_analysis.to_string(index=False))
t40007 = eval_documents.loc[eval_documents["document_id"] == 40007, "text"].iloc[0]
print("40007 원문에 '하나만 더 여쭤볼게요' 등장 횟수:", t40007.count("하나만 더 여쭤볼게요"))
print("예시 40011 원문에 '해지' 포함:", "해지" in example_table.loc[1, "text"])

# 6. 단일 호출과 순차 처리 비교
wf = eval_documents[["document_id", "정답"]].merge(workflow_outputs, on="document_id", validate="one_to_one")
wf["단일_정답"] = wf["단일_분류"] == wf["정답"]
wf["순차_정답"] = wf["순차_분류"] == wf["정답"]
print("\n[문서별 워크플로우 결과]")
print(wf[["document_id", "정답", "단일_분류", "단일_정답", "순차_분류", "순차_정답", "요약_사실일치"]].to_string(index=False))
print("40003 요약(사실 불일치):", wf.loc[wf["document_id"] == 40003, "공통_가상요약"].iloc[0])
print(" -> 원문은 결합 할인 유지, 멤버십은 VIP에서 일반 플러스로 변경이라 두 사실이 반대로 적힘")

calls_per_doc = {"단일 호출": 1, "순차 처리": 2}
workflow_comparison = pd.DataFrame([
    {"방식": "단일 호출", "분류_정답건수": int(wf["단일_정답"].sum()),
     "Accuracy": wf["단일_정답"].sum() / len(wf),
     "요약_사실일치건수": int(wf["요약_사실일치"].sum()),
     "문서당_예상호출수": calls_per_doc["단일 호출"], "전체_예상호출수": calls_per_doc["단일 호출"] * len(wf)},
    {"방식": "순차 처리", "분류_정답건수": int(wf["순차_정답"].sum()),
     "Accuracy": wf["순차_정답"].sum() / len(wf),
     "요약_사실일치건수": int(wf["요약_사실일치"].sum()),
     "문서당_예상호출수": calls_per_doc["순차 처리"], "전체_예상호출수": calls_per_doc["순차 처리"] * len(wf)},
])
print("\n[workflow_comparison]")
print(workflow_comparison.to_string(index=False))

# 7. 우선 검토 방식과 추가 확인 사항
print("\n[우선 검토 의견]")
print("- 프롬프트: Few-shot이 정확도는 높지만(0.4 -> 0.6) 40006에서 새 오류가 생겨, 단일 사례가 중요한 상황이면 예시를 보강한 뒤 다시 비교해야 함")
print("- 워크플로우: 분류 정확도가 중요하고 시간과 사용량에 여유가 있으면 순차 처리를, 호출 수를 줄여야 하면 단일 호출을 우선 검토")
print("- 추가 확인: 같은 조건 반복 실행으로 결과 변동 확인, 평가셋 밖의 다른 상담과 단일 사례를 더 넣어 비교, 실제 처리 시간과 토큰 사용량 측정")


겹치는 ID: set()
겹치는 원문 수: 0
예시 정답 확인:
 document_id   CSV_분류     예시정답                         판단근거
       40002 단일 요건 민원 단일 요건 민원 명의변경 한 요청의 절차·서류·변경 영향·방문 안내
       40011 다수 요건 민원 다수 요건 민원      영화 월정액 해지·요금제 검토·멤버십 신청

[zero_prompt 앞부분]
상담 요건을 단일 요건 민원 또는 다수 요건 민원 중 하나로 분류하세요. 중심 요청·주제가 하나면 단일, 구분되는 요청·주제가 둘 이상이면 다수입니다. 한 요청의 절차·서류·시점을 여러 번 질문한 것만으로 다수로 보지 마세요. 상담 중 추가로 이어진 별도 주제도 포함하여 판단하세요. 범주명만 출력하세요.

[분류할 상담]
상담 원문:
상담사: 일상에 즐거운 변화 LGU+ <NAME>입니다.
고객: 아, 예, 저기 그 해지 때문에 그러는데요.
상담사: 네.
고객: ...

[few_prompt 구성 확인]
두 프롬프트 모두 공통 기준으로 시작: True
두 프롬프트 모두 같은 대상 블록으로 끝남: True
zero_prompt에 예시 원문 없음: True
few_prompt에 예시 원문 두 개 포함: True
few_prompt의 예시 수: 2
글자 수 zero / few: 2655 / 7844
늘어난 부분이 예시 블록과 같음: True

Zero-shot 정답 건수: 2 / 5 / Accuracy: 0.4
Few-shot 정답 건수: 3 / 5 / Accuracy: 0.6

[prompt_comparison]
 document_id       정답 Zero-shot Few-shot  A_정답  B_정답    변화
       40001 다수 요건 민원  다수 요건 민원 다수 요건 민원  True  True 정답 유지
       40003 다수 요건 민원  단일 요건 민원 다수 요건 민원 False  True    개선
       40005 다수

### 해석 답안 작성

- Q1. Zero-shot(A)은 5개 중 2개 정답으로 Accuracy 0.4, Few-shot(B)은 3개 정답으로 0.6이다. 문서별로 보면 40001은 정답 유지, 40003과 40005는 개선, 40006은 새 오류, 40007은 오답 유지다. 전체 정확도는 올랐지만 A에서 맞던 40006(단일 요건)이 B에서 틀리게 되었고, 40007은 여전히 틀렸다. 특히 평가셋에 하나뿐인 단일 사례를 틀리게 된 거라서 단일 요건 구분은 오히려 나빠졌다. 그래서 변경 후 방식이 항상 좋다고 할 수 없고, 가상 자료에 문서도 5개뿐이라 실제 개선 효과라고 단정할 수도 없다.
- Q2. 평가할 문서를 예시로 넣으면 모델이 그 문서의 정답을 프롬프트에서 미리 보고 답하게 되어서 정확도가 실제보다 높게 나온다. 그러면 새로운 상담에서도 잘하는지 알 수 없다. 이번에 확인해 보니 예시 40002, 40011과 평가 문서 5개는 ID도 원문도 겹치지 않았다(겹치는 ID 없음, 겹치는 원문 0개). 그리고 두 방식을 서로 다른 문서로 평가하면 정확도 차이가 프롬프트 때문인지 문서 난이도 때문인지 구분할 수 없다. 그래서 같은 문서, 같은 정답, 같은 기준으로 비교해야 하고, 프롬프트도 공통 기준과 대상 원문은 똑같이 두고 예시 블록만 다르게 했다(늘어난 부분이 예시 블록과 같음 True).
- Q3. 분류는 단일 호출이 3개 정답(0.6), 순차 처리가 4개 정답(0.8)이었다. 차이는 40006 하나로, 단일 호출은 다수로 틀렸고 순차 처리는 단일로 맞혔다. 40007은 두 방식 모두 틀렸다. 요약은 비교를 단순화하려고 같은 문장을 줬기 때문에 사실 일치가 둘 다 4건으로 같았고, 40003 요약만 결합 할인이 해지되고 VIP가 유지됐다고 반대로 적어서 틀렸다. 호출 수는 문서당 1번과 2번, 전체로 5번과 10번이라 두 배지만 실제 시간과 비용이 정확히 두 배라고 할 수 없다. 단계마다 입력과 출력 토큰 수가 다르고(분류 출력은 짧고 요약 출력은 길다), 순차 처리는 원문을 두 번 보내고, 요청 대기 시간이나 재시도도 달라질 수 있고, 이 숫자는 정상 경로 기준 예상 횟수이지 실제 과금 기록이 아니기 때문이다.
- Q4. 개선안을 적용하면 기존과 같은 평가 문서와 정답, 같은 기준으로 다시 비교하고, 전체 정확도뿐 아니라 문서별로 개선, 새 오류, 오답 유지를 다시 확인해야 한다. 반복 실행으로 결과가 흔들리는지도 보고, 평가셋 밖의 새 상담에서도 확인해야 한다. 같은 평가 정답에 맞춰 프롬프트를 계속 고치면 그 5개 문서에만 맞춘 프롬프트가 되어서, 숫자는 오르지만 새로운 상담에서는 효과가 없을 수 있다. 그래서 수정에 쓰는 자료와 최종 검증 자료를 나눠야 한다. 또 출력이 누락되거나 순차 처리에서 중간 분류가 틀리면, 그 문서는 오답이나 실패로 그대로 기록해서 분모에서 빼지 말아야 한다. 중간 분류 오류는 다음 단계 요약까지 영향을 주니까 어느 단계에서 생긴 오류인지 나눠서 기록해야 한다.


---
## 실습 마무리

아래 질문에 각각 2~3문장으로 답하세요.

1. **어떤 분석 문제가 있었는가?** 반복되는 오답, 출력 변동, 잘못된 추출·누락·환각과 개선 후의 새 오류를 정리하세요.
2. **어떤 방법을 적용했는가?** 분류 Accuracy, 필드별 의미 검토, 요약 기준, 동일 데이터 비교와 오류 분석을 설명하세요.
3. **무엇을 근거로 결론을 내렸는가?** 검증된 정답·원문·반복 기록·문서별 전후 비교를 구분하세요.

### 제출 전 확인

- 필수 1·필수 2·과제 1과 Q1~Q4를 모두 작성했나요?
- 정확성과 일관성, 문자열 비교와 의미 판단을 구분했나요?
- 사람이 판단한 평가값을 자동 채점이라고 표현하지 않았나요?
- 같은 문서와 같은 기준으로 비교하고 Few-shot 예시를 분리했나요?
- 개선된 오류뿐 아니라 새 오류와 남은 오류도 확인했나요?
- 가상 결과를 실제 모델 성능이나 검증된 개선 효과로 주장하지 않았나요?


### 마무리 답안

1. 40003은 세 번 모두 단일로 틀려서 일관되게 반복되는 오답이었고, 40005는 2회차에서만 틀려서 출력이 흔들렸다. 정보 추출에서는 과금 여부를 반대로 쓴 잘못된 값과 빈 값 누락이 있었고, 요약 B에는 원문에 없는 고장과 구매(환각)와 할인 가입 왜곡이, C에는 핵심 누락이 있었다. Few-shot으로 바꾼 뒤에는 40003, 40005가 개선됐지만 40006에서 새 오류가 생기고 40007은 계속 틀렸다.
2. 분류는 회차별로 정답 건수를 평가 문서 수 5로 나눠 Accuracy를 구하고, 3회_동일로 일관성을, 3회_모두정답으로 정확성을 따로 봤다. 정보 추출은 코드로 문자열 일치와 항목 존재를 계산하고 사람이 원문을 읽고 의미 일치를 판단했으며, 요약은 사실 일치, 핵심 유지, 원문 밖 정보 세 기준으로 나눠 봤다. 프롬프트와 워크플로우는 같은 문서와 정답으로 비교하고, 문서별 변화 유형을 나눠 오류 원인을 가설로 적었다.
3. 정확성 판단은 원문을 검토해 정한 평가 정답과 비교했고, 일관성 판단은 세 번의 반복 기록끼리 비교했다. 추출과 요약은 원문에 실제로 있는 발화("과금되진 않으세요", "다음에 할게요" 등)를 근거로 사람이 판단했다. 개선 여부는 전체 정확도 대신 문서별 전후 비교로 결론 냈고, 모든 결과가 가상 자료라 실제 성능이라고 하지 않았다.
